In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Fusion baseline: QID embedding + ViT features + MLP (multi-head)

Single shared MLP with per-question heads.


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from tqdm.auto import tqdm

from transformers import AutoImageProcessor, AutoModel
from PIL import Image


2026-02-03 00:34:16.349779: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-03 00:34:16.349817: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-03 00:34:16.350937: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-03 00:34:16.357890: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-03 00:34:17.205100: W tensorflow/compiler/tf2

In [3]:
from pathlib import Path
import os

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "ImageCLEF_MEDVQA_GI_2023" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
sys.path.append(str(ROOT))

from common import (
    find_long_table,
    load_long_table,
    load_label_maps,
    add_label_ids,
    compute_metrics_per_question,
    compute_binary_metrics,
    save_metrics,
    save_predictions,
)

DATA_PATH = find_long_table(ROOT)
LABEL_MAP_DIR = ROOT / "0_dataset_prep" / "out" / "label_maps"
OUT_DIR = ROOT / "1_baselines" / "out" / "03_fusion_qid_vit_mlp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-3
QID_EMB_DIM = 32
HIDDEN_DIM = 256
MAX_SAMPLES_PER_SPLIT = int(os.environ.get("MAX_SAMPLES_PER_SPLIT", "0")) or None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_CACHE = OUT_DIR / "vit_features.npz"


In [4]:
label_maps = load_label_maps(LABEL_MAP_DIR)
long_df = load_long_table(DATA_PATH)

if MAX_SAMPLES_PER_SPLIT:
    long_df = long_df.groupby("split", group_keys=False).head(MAX_SAMPLES_PER_SPLIT)

# Build qid index
qids = sorted(long_df["question_id"].unique())
qid_to_idx = {q: i for i, q in enumerate(qids)}

# Unique images to embed
images_df = long_df[["image_id", "image_path"]].drop_duplicates().reset_index(drop=True)


In [5]:
# Extract or load ViT features
if FEATURE_CACHE.exists():
    data = np.load(FEATURE_CACHE, allow_pickle=True)
    image_ids = data["image_ids"].tolist()
    feats = data["feats"]
else:
    processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)
    model.to(DEVICE)
    model.eval()

    feats = []
    image_ids = []
    for start in tqdm(range(0, len(images_df), BATCH_SIZE), desc="ViT embed"):
        batch = images_df.iloc[start : start + BATCH_SIZE]
        imgs = [Image.open(p).convert("RGB") for p in batch["image_path"].tolist()]
        inputs = processor(images=imgs, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
            if hasattr(out, "pooler_output") and out.pooler_output is not None:
                emb = out.pooler_output
            else:
                emb = out.last_hidden_state[:, 0]
        feats.append(emb.cpu().numpy())
        image_ids.extend(batch["image_id"].tolist())

    feats = np.concatenate(feats, axis=0)
    np.savez(FEATURE_CACHE, image_ids=np.array(image_ids, dtype=object), feats=feats)

id_to_idx = {img_id: i for i, img_id in enumerate(image_ids)}


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


ViT embed:   0%|          | 0/63 [00:00<?, ?it/s]

In [6]:
# Build dataset
feat_mat = np.stack([feats[id_to_idx[i]] for i in long_df["image_id"].tolist()])

long_df = add_label_ids(long_df, label_maps, ans_col="answer_norm", out_col="label_id")

class FusionDataset(Dataset):
    def __init__(self, df, feats):
        self.df = df.reset_index(drop=True)
        self.feats = feats

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "feat": torch.tensor(self.feats[idx]).float(),
            "qid": torch.tensor(qid_to_idx[row["question_id"]]).long(),
            "label": torch.tensor(int(row["label_id"])).long(),
            "split": row["split"],
            "question_id": row["question_id"],
        }

train_mask = long_df["split"] == "train"
train_ds = FusionDataset(long_df[train_mask], feat_mat[train_mask.values])
val_ds = FusionDataset(long_df[~train_mask], feat_mat[~train_mask.values])


In [7]:
# Model
class FusionMLP(nn.Module):
    def __init__(self, img_dim, qid_count, qid_emb_dim, hidden_dim, head_sizes):
        super().__init__()
        self.qid_emb = nn.Embedding(qid_count, qid_emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(img_dim + qid_emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.heads = nn.ModuleDict({str(k): nn.Linear(hidden_dim, v) for k, v in head_sizes.items()})

    def forward_base(self, feat, qid_idx):
        q = self.qid_emb(qid_idx)
        x = torch.cat([feat, q], dim=1)
        return self.mlp(x)

head_sizes = {qid: len(label_maps[str(qid)]["answer_to_id"]) for qid in qids}
model = FusionMLP(feat_mat.shape[1], len(qids), QID_EMB_DIM, HIDDEN_DIM, head_sizes).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()


In [8]:
# Train
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        feat = batch["feat"].to(DEVICE)
        qid_idx = batch["qid"].to(DEVICE)
        label = batch["label"].to(DEVICE)
        qid_raw = batch["question_id"]

        opt.zero_grad()
        h = model.forward_base(feat, qid_idx)

        loss = 0.0
        for qid in set(qid_raw):
            mask = [i for i, q in enumerate(qid_raw) if q == qid]
            if not mask:
                continue
            idx = torch.tensor(mask, device=DEVICE)
            logits = model.heads[str(qid)](h[idx])
            loss = loss + criterion(logits, label[idx])
        loss = loss / max(1, len(set(qid_raw)))
        loss.backward()
        opt.step()
        total_loss += float(loss.item())
    print(f"epoch {epoch+1}/{EPOCHS} loss={total_loss/len(train_loader):.4f}")


epoch 1/5 loss=0.4504
epoch 2/5 loss=0.3032
epoch 3/5 loss=0.2756
epoch 4/5 loss=0.2568
epoch 5/5 loss=0.2405


In [9]:
# Predict
model.eval()
pred_label_ids = np.full(len(long_df), -1, dtype=int)

with torch.no_grad():
    for qid, g in long_df.groupby("question_id"):
        idx = g.index.values
        feat = torch.tensor(feat_mat[idx]).float().to(DEVICE)
        qid_idx = torch.full((len(idx),), qid_to_idx[qid], dtype=torch.long, device=DEVICE)
        h = model.forward_base(feat, qid_idx)
        logits = model.heads[str(qid)](h)
        pred = torch.argmax(logits, dim=1).cpu().numpy()
        pred_label_ids[idx] = pred

pred_df = long_df.copy()
pred_df["pred_label_id"] = pred_label_ids


In [10]:
for split in sorted(pred_df["split"].unique()):
    df_split = pred_df[pred_df["split"] == split]
    overall, per_q = compute_metrics_per_question(df_split, "label_id", "pred_label_id")
    binary = compute_binary_metrics(df_split, label_maps, "label_id", "pred_label_id")
    split_out = OUT_DIR / split
    save_metrics(split_out, overall, per_q, binary)
    save_predictions(
        df_split,
        split_out,
        columns=["image_id", "question_id", "label_id", "pred_label_id", "split"],
    )

OUT_DIR


PosixPath('/home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/1_baselines/out/03_fusion_qid_vit_mlp')